**Imports**

In [34]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50


**Load the CLEAN dataset from TFDS**

In [35]:
train_ds, val_ds = tfds.load(
    "cats_vs_dogs",
    split=["train[:80%]", "train[80%:]"],
    as_supervised=True
)


**Preprocess (Resize + Normalize)**

In [36]:
IMG_SIZE = 224

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0
    return image, label

train_ds = train_ds.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)


**Evaluate ImageNet accuracy BEFORE transfer learning**

In [37]:
labels

<tf.Tensor: shape=(12,), dtype=int64, numpy=array([1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0])>

In [38]:
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
import tensorflow as tf

# Load ImageNet pretrained model
model = ResNet50(weights="imagenet")

correct = 0
total = 0

for images, labels in val_ds:
    # images are already 224×224 and normalized (0–1)
    imgs = preprocess_input(images * 255.0)  # convert to ImageNet format

    # Predict
    preds = model.predict(imgs, verbose=0)
    pred_idxs = tf.argmax(preds, axis=1).numpy()  # 0–999 ImageNet classes

    # Compare directly with your labels (0 or 1)
    for p, true in zip(pred_idxs, labels.numpy()):
        if p == true:
            correct += 1
        total += 1

accuracy = correct / total
print("ImageNet BEFORE transfer learning accuracy:", accuracy)


ImageNet BEFORE transfer learning accuracy: 0.0


**Build the ResNet50 Transfer Learning Model**

In [39]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,#Remove the original classifier from the model.
    input_shape=(224, 224, 3)
)

base_model.trainable = False  # freeze backbone

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(2, activation="softmax")
])


**Compile**

In [40]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


**Train**

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


Epoch 1/5
279/582 ━━━━━━━━━━━━━━━━━━━━ 12:14 2s/step - accuracy: 0.5553 - loss: 0.7072

**Evaluate**

In [ ]:
loss, acc = model.evaluate(val_ds)
print("Validation accuracy:", acc)
